In [ ]:
!pip install yt-dlp requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.8 MB/s eta 0:00:00


In [ ]:
import csv
import re
import time
import subprocess
from pathlib import Path
import requests
import concurrent.futures
from threading import Semaphore
import json
import html
import os

# ─────────────────────────────
# CONFIGURAÇÕES
# ─────────────────────────────
IDIOMA = "pt"
IDIOMA_FALLBACK = "en"
FORMATO = "vtt"  # ou "srt", "json3" etc.
ARQUIVO_CSV = "transcripts_ytdlp.csv"
MAX_VIDEOS = 10
BUSCA = "machine learning"
MAX_THREADS = 5

# ─────────────────────────────
# FUNÇÕES
# ─────────────────────────────

def buscar_videos_youtube(query, max_resultados=10):
    """Busca vídeos no YouTube usando API não-oficial com tratamento robusto de caracteres."""
    try:
        search_url = f"https://www.youtube.com/results?search_query={requests.utils.quote(query)}&hl={IDIOMA}"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
            'Accept-Language': f'{IDIOMA};q=0.9',
        }
        response = requests.get(search_url, headers=headers, timeout=15)
        response.raise_for_status()

        pattern = r'var ytInitialData\s*=\s*({.*?});'
        match = re.search(pattern, response.text, re.DOTALL)
        if not match:
            print("⚠️ Estrutura de dados não encontrada na página")
            return []

        data = json.loads(match.group(1))
        videos = []
        contents = data.get('contents', {}) \
                       .get('twoColumnSearchResultsRenderer', {}) \
                       .get('primaryContents', {}) \
                       .get('sectionListRenderer', {}) \
                       .get('contents', [])

        for section in contents:
            items = section.get('itemSectionRenderer', {}).get('contents', [])
            for item in items:
                if 'videoRenderer' in item:
                    vr = item['videoRenderer']
                    vid = vr.get('videoId')
                    # trata HTML entities no título
                    runs = vr.get('title', {}).get('runs', [])
                    title = ''.join(r.get('text','') for r in runs) if runs else vr.get('title', {}).get('simpleText','')
                    title = html.unescape(title)
                    if vid and title:
                        videos.append({'id': vid, 'title': title})
                    if len(videos) >= max_resultados:
                        break
            if len(videos) >= max_resultados:
                break
        return videos

    except Exception as e:
        print(f"🚨 Erro na busca de vídeos: {e}")
        return []

def extrair_texto_legenda(path: Path):
    """Extrai texto limpo de legendas VTT ou SRT."""
    linhas = []
    with open(path, 'r', encoding='utf-8') as f:
        for l in f:
            t = l.strip()
            if not t:
                continue
            # ignora timestamps, índices e cabeçalho VTT
            if re.match(r'^\d+$', t) or '-->' in t or t.upper().startswith("WEBVTT"):
                continue
            linhas.append(t)
    return " ".join(linhas)

def processar_video(video, query, semaforo):
    """Processa um único vídeo: baixa legenda .vtt e extrai texto."""
    with semaforo:
        vid = video['id']
        titulo = video['title']
        resultado = {'video_id': vid, 'titulo': titulo, 'query': query, 'idioma': IDIOMA, 'transcript': ""}

        print(f"▶ Processando: {titulo[:60]}{'...' if len(titulo)>60 else ''}")
        cmd = [
            "yt-dlp",
            "--write-auto-sub",  # legendas automáticas
            "--write-sub",       # legendas manuais (se houver)
            "--skip-download",
            "--sub-lang", f"{IDIOMA},{IDIOMA_FALLBACK}",
            "--output", vid,
            "--no-warnings",
            f"https://www.youtube.com/watch?v={vid}"
        ]
        try:
            subprocess.run(cmd, check=True, capture_output=True, text=True, timeout=90)

            # tenta VTT em português
            glob_pt = Path(".").glob(f"{vid}.{IDIOMA}.*.{FORMATO}")
            legenda = next(glob_pt, None)

            # se não achou, tenta idioma fallback
            if not legenda or not legenda.exists():
                glob_en = Path(".").glob(f"{vid}.{IDIOMA_FALLBACK}.*.{FORMATO}")
                legenda = next(glob_en, None)

            # última opção: qualquer formato .vtt/.srt
            if not legenda or not legenda.exists():
                legenda = next(Path(".").glob(f"{vid}.*.{FORMATO}"), None)

            if legenda and legenda.exists():
                texto = extrair_texto_legenda(legenda)
                resultado['transcript'] = texto[:100000]
                legenda.unlink()
                status = f"✅ {len(resultado['transcript'])} caracteres"
            else:
                status = "⚠️ Legenda não encontrada"

            print(f"  {status}")

        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
            err = e.stderr or e.stdout or str(e)
            print(f"  🚫 Erro yt-dlp: {err[:100]}{'...' if len(err)>100 else ''}")
        except Exception as e:
            print(f"  ❌ Erro inesperado: {e}")

        return resultado

def processar_videos_paralelo(query, max_videos=5, arquivo_csv="saida.csv"):
    print(f"🔍 Buscando vídeos para: '{query}'")
    videos = buscar_videos_youtube(query, max_resultados=max_videos)
    if not videos:
        print("❌ Nenhum vídeo encontrado.")
        return

    print(f"📥 {len(videos)} vídeos encontrados")
    sem = Semaphore(MAX_THREADS)
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_THREADS) as ex:
        futures = [ex.submit(processar_video, v, query, sem) for v in videos]
        resultados = [f.result() for f in concurrent.futures.as_completed(futures)]

    validos = [r for r in resultados if r['transcript'] or r['video_id'] != 'erro']
    modo = 'a' if os.path.exists(arquivo_csv) else 'w'
    with open(arquivo_csv, modo, newline='', encoding='utf-8') as csvfile:
        w = csv.DictWriter(csvfile, fieldnames=["video_id","titulo","query","idioma","transcript"])
        if modo=='w': w.writeheader()
        w.writerows(validos)

    checados = sum(1 for r in validos if r['transcript'].strip())
    print(f"\n✅ Concluído: {checados}/{len(videos)} com transcript")
    print(f"💾 Salvo em: {arquivo_csv}")

# ─────────────────────────────
# EXECUÇÃO
# ─────────────────────────────
if __name__ == "__main__":
    inicio = time.time()
    try:
        processar_videos_paralelo(BUSCA, max_videos=MAX_VIDEOS, arquivo_csv=ARQUIVO_CSV)
    finally:
        print(f"⏱️ Tempo total: {time.time() - inicio:.2f}s")


🔍 Buscando vídeos para: 'machine learning'
📥 10 vídeos encontrados
▶ Processando: Machine Learning for Everybody – Full Course
▶ Processando: Stanford CS229 I Machine Learning I Building Large Language ...
▶ Processando: PyTorch for Deep Learning & Machine Learning – Full Course
▶ Processando: Complete Machine Learning In 6 Hours| Krish Naik
▶ Processando: Machine Learning | What Is Machine Learning? | Introduction ...
  ✅ 100000 caracteres
▶ Processando: All Machine Learning algorithms explained in 17 min
  ✅ 51200 caracteres
▶ Processando: Essential Machine Learning and AI Concepts Animated
  ✅ 100000 caracteres
▶ Processando: All Machine Learning Models Clearly Explained!
  ✅ 100000 caracteres
▶ Processando: Artificial Intelligence (AI) and Machine Learning (ML)
  ✅ 100000 caracteres
▶ Processando: Machine Learning in 2024 – Beginner's Course
  ✅ 100000 caracteres
  ✅ 100000 caracteres
  ✅ 9435 caracteres
  ✅ 100000 caracteres
  ✅ 100000 caracteres

✅ Concluído: 10/10 com transcript

In [ ]:
import requests
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random
import os

# Configuração: Número desejado de proxies válidos
TARGET_VALID = 20  # Altere este valor conforme necessário

# Lista de URLs de proxies (mantida igual)
PROXY_SOURCES = [
    "https://raw.githubusercontent.com/hendrikbgr/Free-Proxy-Repo/master/proxy_list.txt",
    "https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/http.txt",
    "https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/socks4.txt",
    "https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/socks5.txt",
    "https://raw.githubusercontent.com/noctiro/getproxy/master/file/http.txt",
    "https://raw.githubusercontent.com/noctiro/getproxy/master/file/https.txt",
    "https://raw.githubusercontent.com/noctiro/getproxy/master/file/socks4.txt",
    "https://raw.githubusercontent.com/noctiro/getproxy/master/file/socks5.txt",
    "https://raw.githubusercontent.com/mmpx12/proxy-list/master/http.txt",
    "https://raw.githubusercontent.com/mmpx12/proxy-list/master/https.txt",
    "https://raw.githubusercontent.com/mmpx12/proxy-list/master/socks4.txt",
    "https://raw.githubusercontent.com/mmpx12/proxy-list/master/socks5.txt",
]

def extract_proxies_from_text(text):
    pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}:\d+\b'
    return re.findall(pattern, text)

def fetch_proxies_from_source(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        proxies = extract_proxies_from_text(response.text)
        print(f"✅ Encontrados {len(proxies)} proxies em {url}")
        return proxies
    except Exception as e:
        print(f"⚠️ Erro ao buscar proxies de {url}: {str(e)}")
        return []

def get_all_proxies():
    all_proxies = []
    for url in PROXY_SOURCES:
        proxies = fetch_proxies_from_source(url)
        all_proxies.extend(proxies)
    unique_proxies = list(set(all_proxies))
    print(f"\n🔍 Total de proxies únicos encontrados: {len(unique_proxies)}")
    return unique_proxies

def test_proxy(proxy):
    test_urls = [
        "http://httpbin.org/ip",
        "https://httpbin.org/ip",
        "http://google.com",
        "https://google.com"
    ]
    proxies_http = {"http": f"http://{proxy}", "https": f"http://{proxy}"}

    # Testa protocolos HTTP/HTTPS
    try:
        url = random.choice(test_urls)
        response = requests.get(url, proxies=proxies_http, timeout=10)
        if response.status_code == 200:
            return True, proxy, "http"
    except:
        pass

    # Testa SOCKS
    for protocol in ["socks4", "socks5"]:
        try:
            proxies_socks = {"http": f"{protocol}://{proxy}", "https": f"{protocol}://{proxy}"}
            url = random.choice(test_urls)
            response = requests.get(url, proxies=proxies_socks, timeout=10)
            if response.status_code == 200:
                return True, proxy, protocol
        except:
            continue
    return False, proxy, None

def test_proxies(proxies, max_workers=100):
    valid_proxies = []
    total = len(proxies)
    print(f"\n🚀 Testando {total} proxies. Alvo: {TARGET_VALID} válidos...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Mapeia cada future para seu proxy correspondente
        future_to_proxy = {executor.submit(test_proxy, proxy): proxy for proxy in proxies}

        try:
            for i, future in enumerate(as_completed(future_to_proxy)):
                proxy = future_to_proxy[future]
                try:
                    success, proxy, protocol = future.result()
                    if success:
                        valid_proxies.append((proxy, protocol))
                        # Verifica se atingiu o alvo
                        if len(valid_proxies) >= TARGET_VALID:
                            print(f"🎯 Atingido {TARGET_VALID} proxies válidos!")
                            break
                except Exception as e:
                    pass

                # Atualiza progresso a cada 100 testes
                if (i + 1) % 100 == 0 or (i + 1) == total:
                    print(f"🔎 Testados: {i+1}/{total} | Válidos: {len(valid_proxies)}")
        finally:
            # Cancela todas as tarefas pendentes se o alvo foi atingido
            for future in future_to_proxy:
                future.cancel()

    print(f"✅ Total de válidos encontrados: {len(valid_proxies)}")
    return valid_proxies

def save_valid_proxies(valid_proxies, filename="proxies_validos.txt"):
    with open(filename, "w") as f:
        for proxy, protocol in valid_proxies:
            f.write(f"{protocol}://{proxy}\n")
    print(f"\n💾 Proxies salvos em: {os.path.abspath(filename)}")

def main():
    print("🚀 Iniciando coleta de proxies...")
    start_time = time.time()

    all_proxies = get_all_proxies()
    valid_proxies = test_proxies(all_proxies)
    save_valid_proxies(valid_proxies)

    print(f"⏱️ Tempo total: {time.time() - start_time:.2f}s")

if __name__ == "__main__":
    main()

🚀 Iniciando coleta de proxies...
✅ Encontrados 590 proxies em https://raw.githubusercontent.com/hendrikbgr/Free-Proxy-Repo/master/proxy_list.txt
✅ Encontrados 38964 proxies em https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/http.txt
✅ Encontrados 2815 proxies em https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/socks4.txt
✅ Encontrados 2096 proxies em https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/socks5.txt
✅ Encontrados 40993 proxies em https://raw.githubusercontent.com/noctiro/getproxy/master/file/http.txt
✅ Encontrados 2213 proxies em https://raw.githubusercontent.com/noctiro/getproxy/master/file/https.txt
✅ Encontrados 7189 proxies em https://raw.githubusercontent.com/noctiro/getproxy/master/file/socks4.txt
✅ Encontrados 5051 proxies em https://raw.githubusercontent.com/noctiro/getproxy/master/file/socks5.txt
✅ Encontrados 530 proxies em https://raw.githubusercontent.com/mmpx12/proxy-list/master/http.txt
✅ Encontrados 203 proxies em https:/